[2025-12-09] - [Emil] - [Skeleton API Deployment]

**What I worked on:**
- I implemented `/deployment/app.py` utilizing the FastAPI framework to build a service that accepts feature vectors as input and returns a prediction of DoS or benign traffic.
- The best performing model was trained and saved to disk as a `.pkl` file - this model was `random_forest.pkl`.  

Below is the code stored in the `deployment/app.py` file:

```
from fastapi import FastAPI, HTTPException
import joblib
import json
from pathlib import Path

from deployment.schema_loader import load_schema
from deployment.preprocessing import Preprocessor

# First we create a new FastAPI object
app = FastAPI(
    title = "DoS Classifier API",
    version = "0.1.0",
    description = "API for predicting DoS attacks vs. benign traffic"
)

# Load the schema
SCHEMA_PATH = Path("data/final_features.json")
schema = load_schema(SCHEMA_PATH)

# Then we create the preprocessor object.
preprocessor = Preprocessor(schema_path = SCHEMA_PATH)

# Load the trained model
MODEL_PATH = Path("models/saved_model/random_forest.pkl")

# This logic is required because as of the writing of this code, no model has been trained. 
if MODEL_PATH.exists():
    model = joblib.load(MODEL_PATH)
else:
    model = None

# Implementation of health check endpoint
@app.get("/health")
def health_check():
    return {"status": "ok"}

# Implementation of version endpoint
@app.get("/version")
def version():
    return {"api_version": "0.1.0"}

# Implementation of prediction endpoint (inference not yet implemented)
@app.post("/predict")
def predict(input_data: dict):
    # Running preprocessing with sanity checking.
    try:
        X = preprocessor.preprocess_input(input_data)
    except Exception as e:
        raise HTTPException(
            status_code = 400, 
            detail = f"Failed to preprocess input data: {e}"
        )
    try:
        y_pred = model.predict(X)
    except Exception as e:
        raise HTTPException(
            status_code = 500, 
            detail = f"Failed to make predictions: {e}"
        )
    return {"prediction": int(y_pred[0])}
```

**Interpretation / Findings:**
In this script, the deployment layer is implemented of our intrusion-detection classifier. It loads the project's `final_features.json` schema and the best-performing trained model to create a reproducible inference pipeline. Incoming rquests to the `/predict` endpoint are validated to ensure correct feature names, types, and ordering. The API then applies the same preprocessing steps used in training, passes the transformed input to the loaded classifier, and returns the predicted class label. I also implemented two endpoints, `/health` and `/version`, which serve as health checks and version endpoints, respectively - entirely diagnostic. 

**Notes for Team:**
This is testable code - we can begin to write unit/Postman tests.

### Done with Task

[2025-12-09] - [Emil] - [Preprocessing for API]

**What I worked on:**
- I implemented the `preprocess_input` method in the `Preprocessor` class to transform incoming feature vectors into a format that can be consumed by the trained model. This file is found in `deployment/preprocessing.py`.

Below is the code stored in the `deployment/preprocessing.py` file:

```
import json
import numpy as np
from pathlib import Path

class Preprocessor:
    def __init__(self, schema_path: str):
        """
        Instantiator method for the preprocessor class. 
        
        :param self: This object.
        :param schema_path: The path to the final_features.json file.
        """
        self.schema_path = Path(schema_path)

        # First we need to load the schema
        with open(self.schema_path, "r") as f:
            schema = json.load(f)
        
        # Then we need to extract the scaler parameters from final_features.json
        scaler_params = schema["scaler_params"]

        self.feature_names = scaler_params["feature_names"]
        self.feature_means = np.array(scaler_params["feature_means"], dtype = float)
        self.feature_stds = np.array(scaler_params["feature_stds"], dtype = float)

        # Convert the numeric types to a dictionary for faster lookup. 
        self.numeric_types = {
            k: v["dtype"]
            for k, v in self.schema.items()
            if isinstance(v, dict) and v.get("feature_type") == "numeric"
        }
        
        self.n_features = len(self.feature_names)

    def validate_input(self, sample: dict):
        # Sanity check to ensure the feature names, means, and stds are the same length
        if len(self.feature_names) != len(self.feature_means) or len(self.feature_names) != len(self.feature_stds):
            raise ValueError("Feature names, means, and stds are not the same length")

        # Check for missing fields. 
        for f in self.features:
            if not f in sample:
                raise ValueError(f"Missing feature {f}")
        
        # Check for unexpected fields.
        for k in sample.keys():
            if k not in self.features:
                raise ValueError(f"Unexpected feature {k}")
        
        # Validate the numeric types.
        for key, expected_type in self.numeric_types.items():
            value = sample[key]
            
            # Raise exception if the value is null.
            if value is None:
                raise ValueError(f"Field '{key}' cannot be null.")
            
            # Try to typecast the value to numeric, otherwise raise exception.
            if not isinstance(value, (int, float)):
                try:
                    sample[key] = float(value)
                except:
                    raise ValueError(f"Field '{key}' must be numeric, but value is '{value}' with type {type(value)}.")
                
        return sample
    
    def safe_scaling(self, X):
        """
        Safely scales the dataset by avoiding divide by zero errors.
        
        :param self: This object.
        :param X: The dataset to be scaled (numpy array). 
        """
        # If the std is too small, replace it with 1.0
        safe_stds = np.where(self.stds < 1e-12, 1.0, self.stds)

        # Apply the scaling.
        return (X - self.means) / safe_stds

    def preprocess_input(self, sample: dict):
        """
        Takes a dictionary containing raw feature inputs
        and returns a (1, n_features) scaled numpy array. 

        Missing features are filled with 0.0.
        
        :param self: This object.
        :param sample: The dictionary containing raw feature inputs.
        """
        # First validate the input
        sample = self.validate(sample)

        # Then order the features.
        ordered = [sample[f] for f in self.features]

        # Then we convert to a dataframe and scale and reshape the array for prediction.
        X = np.array(ordered, dtype = float).reshape(1, -1)
        
        # Apply scaling.
        X_scaled = self.scale(X)

        return X_scaled
```

**Interpretation / Findings:**
This class is responsible for enforcing strict input validation and for preparing incoming JSON data for inference. When a request arrives at the `/predict` endpoint, the preprocessor loads the project's feature schema stored in `final_features.json`. We do this to make sure that input contains all expected features, includes no unexpected features, and every value is numeric and non-null (performing coercion where possible). After validation, features are reordered to exactly match layout used during model training. The preprocessor then applies the same scaling applied during training. The entire preprocessing pipeline then returns a NumPy array ready to be passed into the trained model. 

**Notes for Team:**
Also testable code!!!